# Thermo Scientific Multidrop Combi nL

The Multidrop Combi nL is a **valve-based, per-well** bulk reagent dispenser. A pressurized reagent reservoir feeds **8 solenoid microvalves** (one per row); volume is metered by valve-open time. Unlike the peristaltic Multidrop Combi — which dispenses a whole column at a time — the nL addresses **every well individually**, across 96-, 384-, and 1536-well plates, from **50 nL to 50 µL**. It connects over **RS232 serial**.

PLR exposes it as a {class}`~pylabrobot.thermo_fisher.multidrop_combi.multidrop_combi_nl.MultidropCombiNl` device with a {class}`~pylabrobot.capabilities.bulk_dispensers.valve.valve8.ValveDispensing8` capability (per-well volumes), driven by {class}`~pylabrobot.thermo_fisher.multidrop_combi.multidrop_combi_nl_backend.MultidropCombiNlValveDispensingBackend8`.

```{warning}
`pressurize()`, `prime()`, `purge()`, and `dispense()` move fluid or pressurize the instrument. Make sure a reagent bottle and waste are in place before running them.
```

## Setup

The nL connects over RS232. Pass the serial port your unit is on — e.g. `/dev/ttyUSB0` on Linux (a USB-to-serial adapter) or `COM3` on Windows. On Linux, adding your user to the `dialout` group grants access without `sudo`.

In [ ]:
from pylabrobot.thermo_fisher.multidrop_combi import MultidropCombiNl

md = MultidropCombiNl(port="/dev/ttyUSB0")  # replace with your unit's serial port
await md.setup()

info = md.driver.get_version()
print(f"{info['instrument_name']} FW {info['firmware_version']} SN {info['serial_number']}")

## Assign a plate

Assign the target plate to the capability. Per-well addressing works for 96-, 384-, and 1536-well plates.

In [ ]:
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb

plate = Cor_96_wellplate_360ul_Fb("my_plate")
md.valve_dispenser.plate = plate

## Plate geometry (needed when changing format)

The nL has **no RFID auto-detection**, so it keeps whatever plate geometry was last configured. When you switch plate format (e.g. from 96 to 384 or 1536), set the geometry explicitly. The instrument ships factory plate definitions — read them and apply the matching one via {meth}`~pylabrobot.thermo_fisher.multidrop_combi.multidrop_combi_nl_backend.MultidropCombiNlValveDispensingBackend8.define_plate`.

In [ ]:
backend = md.valve_dispenser.backend

for p in await backend.get_factory_plates():
    print(p["name"], f"({p['columns']}x{p['rows']}, {p['height']/100:.1f}mm)")

# Configure, e.g., 384 standard by matching the factory name:
# match = next(p for p in await backend.get_factory_plates() if "384&standard" in p["name"])
# await backend.define_plate(match["pla"])

## Pressurize and prime

The nL dispenses from a **pressurized** reservoir. Pressurize, then prime — priming is *hold-to-run*, so pass a `duration` (seconds) to fill the tubing from dry.

In [ ]:
await backend.pressurize()          # PON 1
await md.valve_dispenser.prime(duration=8)   # hold-prime ~8 s

## Per-well dispensing

Volumes may be given as:

* a single number — the same volume into **every** well;
* a dict keyed by **well name** (`{"A1": 5.0, "B1": 2.5}`); or
* a dict keyed by 1-indexed **`(row, col)`** (`{(1, 1): 5.0}`).

Volumes are in µL (0.05–50 µL). **Variable volumes per well** are fully supported — each distinct volume is metered independently in a single dispense (great for gradients / dilution series).

In [ ]:
# 2 uL into every well
await md.valve_dispenser.dispense(2.0)

# Per-well by name: a dilution series down column 1 (variable volumes)
await md.valve_dispenser.dispense({"A1": 5.0, "B1": 2.5, "C1": 1.0, "D1": 0.5})

# Per-well by (row, col)
await md.valve_dispenser.dispense({(1, 1): 5.0, (1, 2): 5.0, (2, 1): 1.0})

Use {class}`~pylabrobot.thermo_fisher.multidrop_combi.multidrop_combi_nl_backend.MultidropCombiNlValveDispensingBackend8.DispenseParams` for geometry: **drop height** (`dispense_height`, in 1/100 mm — e.g. `2000` = 20 mm tip-to-plate), X/Y `offset`, `dispensing_order`, and the calibration `pressure_index`.

In [ ]:
from pylabrobot.thermo_fisher.multidrop_combi import MultidropCombiNlValveDispensingBackend8

await md.valve_dispenser.dispense(
    1.0,
    backend_params=MultidropCombiNlValveDispensingBackend8.DispenseParams(dispense_height=2000),
)

## Purge, release pressure, teardown

Purge the lines, release the reservoir pressure, then stop (which sends `QIT` and closes the connection).

In [ ]:
await md.valve_dispenser.purge()          # EMP
await backend.release_pressure()          # POF
await md.stop()